# MTG Deck Predictor — Optuna Hyperparameter Sweep

Automated search for optimal hyperparameters using Bayesian optimization.
Optuna tries many configurations, prunes unpromising trials early, and reports
which parameters matter most.

**Runtime**: Select `Runtime > Change runtime type > T4 GPU` before running.

**Expected time**: ~2–3 hours for 30 trials on T4.

In [ ]:
# Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/mtg-graph'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

!pip install -q torch-geometric sentence-transformers python-dotenv \
    beautifulsoup4 lxml spacy tqdm pyarrow optuna

print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')

In [ ]:
# ── Search Space Configuration (edit these) ──

N_TRIALS = 30              # Total trials to run (more = better search, slower)
STUDY_NAME = 'deck-sweep'  # Name for the Optuna study

# Search ranges — Optuna samples from these each trial
SEARCH_SPACE = {
    'hidden_dim':     [32, 64, 128],           # categorical
    'num_hgt_layers': (1, 3),                  # int range (inclusive)
    'dropout':        (0.1, 0.5, 0.05),        # float range (min, max, step)
    'learning_rate':  (1e-4, 1e-2),            # float range (log scale)
    'weight_decay':   (1e-5, 1e-2),            # float range (log scale)
    'n_negatives':    [25, 50, 100],           # categorical
    'warmup_epochs':  [0, 3, 5, 10],           # categorical
    'lr_scheduler':   ['cosine', 'linear', 'none'],  # categorical
}

# Fixed across all trials (not searched)
FIXED_PARAMS = {
    'num_heads': 4,
    'bpr_margin': 1.0,
    'patience': 10,
    'num_epochs': 50,
    'recency_days': 30,
    'checkpoint_metric': 'val_hits10',
}

# Pruning — kill underperforming trials early
N_STARTUP_TRIALS = 5  # Run this many trials to completion before pruning
N_WARMUP_STEPS = 2    # Don't prune until trial has reported this many eval steps

print(f'Search: {N_TRIALS} trials over {len(SEARCH_SPACE)} parameters')
print(f'Pruning: starts after {N_STARTUP_TRIALS} trials, warmup {N_WARMUP_STEPS} steps')
print(f'Fixed: {FIXED_PARAMS}')

## Run Sweep

In [ ]:
import optuna
import time
import logging

# Reduce Optuna's verbose logging
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger('src.train_deck').setLevel(logging.WARNING)

from src.train_deck import train_deck


def objective(trial):
    """Optuna objective: sample hyperparameters, train, return best Hits@10."""
    params = dict(FIXED_PARAMS)  # start with fixed

    # Sample from search space
    params['hidden_dim'] = trial.suggest_categorical('hidden_dim', SEARCH_SPACE['hidden_dim'])
    params['num_hgt_layers'] = trial.suggest_int('num_hgt_layers', *SEARCH_SPACE['num_hgt_layers'])
    params['dropout'] = trial.suggest_float('dropout', *SEARCH_SPACE['dropout'][:2], step=SEARCH_SPACE['dropout'][2])
    params['learning_rate'] = trial.suggest_float('learning_rate', *SEARCH_SPACE['learning_rate'], log=True)
    params['weight_decay'] = trial.suggest_float('weight_decay', *SEARCH_SPACE['weight_decay'], log=True)
    params['n_negatives'] = trial.suggest_categorical('n_negatives', SEARCH_SPACE['n_negatives'])
    params['warmup_epochs'] = trial.suggest_categorical('warmup_epochs', SEARCH_SPACE['warmup_epochs'])
    params['lr_scheduler'] = trial.suggest_categorical('lr_scheduler', SEARCH_SPACE['lr_scheduler'])

    start = time.time()
    result = train_deck(device=device, trial=trial, **params)
    elapsed = time.time() - start

    best_metric = result['best_val_metric']
    print(f'  Trial {trial.number}: Hits@10={best_metric:.4f} '
          f'(epoch {result["best_epoch"]}, {elapsed:.0f}s) '
          f'| hd={params["hidden_dim"]} ly={params["num_hgt_layers"]} '
          f'lr={params["learning_rate"]:.1e} do={params["dropout"]:.2f}')

    return best_metric


# Create study with median pruner
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=N_STARTUP_TRIALS,
    n_warmup_steps=N_WARMUP_STEPS,
)
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    pruner=pruner,
)

print(f'Starting {N_TRIALS}-trial sweep...\n')
sweep_start = time.time()
study.optimize(objective, n_trials=N_TRIALS)
sweep_elapsed = time.time() - sweep_start

print(f'\n{"=" * 60}')
print(f'Sweep complete: {len(study.trials)} trials in {sweep_elapsed/60:.1f} min')
print(f'  Pruned: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}')
print(f'  Completed: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}')
print(f'\nBest trial #{study.best_trial.number}:')
print(f'  Hits@10 = {study.best_value:.4f}')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
import matplotlib.pyplot as plt

# Optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Trial values over time
ax = axes[0]
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_nums = [t.number for t in completed]
trial_vals = [t.value for t in completed]
best_so_far = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]

ax.scatter(trial_nums, trial_vals, alpha=0.5, s=20, label='Trial value')
ax.plot(trial_nums, best_so_far, 'r-', linewidth=2, label='Best so far')

# Mark pruned trials
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
if pruned:
    ax.scatter([t.number for t in pruned], [0] * len(pruned),
              marker='x', color='gray', alpha=0.3, s=15, label='Pruned')

ax.set_xlabel('Trial'); ax.set_ylabel('Hits@10')
ax.set_title('Optimization History'); ax.legend(); ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[1]
try:
    importance = optuna.importance.get_param_importances(study)
    names = list(importance.keys())
    values = list(importance.values())
    colors = plt.cm.viridis([v / max(values) for v in values])
    ax.barh(names, values, color=colors)
    ax.set_xlabel('Importance'); ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except Exception as e:
    ax.text(0.5, 0.5, f'Importance unavailable:\n{e}',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

# Print all completed trials sorted by value
print(f'\nAll completed trials (sorted by Hits@10):')
print(f'{"#":>4}  {"Hits@10":>8}  {"hd":>4}  {"ly":>3}  {"lr":>10}  {"do":>5}  {"wd":>10}  {"neg":>4}  {"wu":>3}  {"sched":>7}')
print('-' * 75)
for t in sorted(completed, key=lambda t: t.value, reverse=True):
    p = t.params
    print(f'{t.number:4d}  {t.value:8.4f}  {p["hidden_dim"]:4d}  {p["num_hgt_layers"]:3d}  '
          f'{p["learning_rate"]:10.1e}  {p["dropout"]:5.2f}  {p["weight_decay"]:10.1e}  '
          f'{p["n_negatives"]:4d}  {p["warmup_epochs"]:3d}  {p["lr_scheduler"]:>7s}')

## Retrain with Best Parameters

Run a full training + evaluation + dashboard with the best hyperparameters found by the sweep.

In [ ]:
import time
logging.getLogger('src.train_deck').setLevel(logging.INFO)

best_params = {**FIXED_PARAMS, **study.best_params}
print(f'Retraining with best parameters:')
for k, v in sorted(best_params.items()):
    print(f'  {k}: {v}')
print()

start = time.time()
result = train_deck(device=device, **best_params)
elapsed = time.time() - start
print(f'\nTraining complete in {elapsed:.1f}s')

from src.train_deck import evaluate_deck
eval_results = evaluate_deck(
    model=result['model'],
    data=result['data'],
    arch_pos_cards=result['arch_pos_cards'],
    arch_neg_pool=result['arch_neg_pool'],
    train_cards=result['train_cards'],
    val_cards=result['val_cards'],
    test_cards=result['test_cards'],
    run_dir=result['run_dir'],
    model_path=result['model_path'],
    train_losses=result['train_losses'],
    val_metrics_history=result['val_metrics_history'],
    best_epoch=result['best_epoch'],
    hp=result['hp'],
)

from src.visualize_deck import main as generate_deck_dashboard
generate_deck_dashboard(
    run_dir=result['run_dir'],
    model_path=result['model_path'],
)

print(f"\nAll results saved to: {result['run_dir']}")
!ls -la "{result['run_dir']}"